# 05 — Cross-Sectional XGBoost Ranking Study

This notebook compares three ways to score stocks within each trading date:

- regression on future market-relative return;
- classification of future top-quintile membership;
- learning to rank with `XGBRanker` and ordinal relevance grades.

It is an exploration notebook, not part of the reusable production training harness. It fits on train and evaluates validation only. Keep the committed notebook in this generic state; export materially changed studies to `notebooks/exports/` as HTML before restoring the base notebook.


In [ ]:
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from xgboost import XGBClassifier, XGBRanker, XGBRegressor

RANDOM_SEED = 23
TOP_K = 10
HORIZON = 5
TOP_QUANTILE_THRESHOLD = 0.80

DATA_CUTOFF = date(2025, 12, 31)
TRAIN_START = date(2000, 1, 1)
TRAIN_END = date(2020, 12, 31)
VALIDATION_START = date(2021, 1, 1)
VALIDATION_END = date(2023, 12, 31)
TEST_START = date(2024, 1, 1)
TEST_END = date(2025, 12, 31)

repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)


## Resolve the exploratory universe

The cross-sectional targets need many stocks on each date. This base notebook reads the current Large Cap and Mid Cap configuration files. Change the two paths or replace `TICKERS` with an explicit tuple for a narrower study.


In [ ]:
def configured_tickers(path: Path) -> tuple[str, ...]:
    config = yaml.safe_load(path.read_text(encoding="utf-8"))
    return tuple(item["ticker"] for item in config["symbols"])

universe_directory = repo_root / "src" / "swingtrader" / "configs" / "universes"
TICKERS = tuple(
    dict.fromkeys(
        configured_tickers(universe_directory / "se_large_cap.yml")
        + configured_tickers(universe_directory / "se_mid_cap.yml")
    )
)

len(TICKERS), TICKERS[:5]


## Build the cross-sectional temporal dataset

The existing dataset and split contracts remain the source of truth. The selected supervised task is the continuous five-session percentile target, while the notebook also reads the relative-return and relevance-grade columns generated by the same target set.


In [ ]:
from swingtrader.data.db import resolve_database_engine
from swingtrader.data.features import DEFAULT_FEATURE_SET
from swingtrader.modeling.datasets import (
    CROSS_SECTIONAL_RETURN_PRIMARY_TASK,
    CROSS_SECTIONAL_RETURN_TARGET_SET,
    TemporalDatasetSpec,
    UniverseSpec,
    build_temporal_dataset,
)
from swingtrader.modeling.experiments import FixedTemporalSplitter, TemporalSplitSpec

universe = UniverseSpec(
    name="stockholm_large_mid_cap_exploration",
    version="1",
    provider="yfinance",
    tickers=TICKERS,
)
dataset_spec = TemporalDatasetSpec(
    feature_set=DEFAULT_FEATURE_SET,
    target_set=CROSS_SECTIONAL_RETURN_TARGET_SET,
    task=CROSS_SECTIONAL_RETURN_PRIMARY_TASK,
    universe=universe,
    data_cutoff=DATA_CUTOFF,
)
split_spec = TemporalSplitSpec(
    name="cross_sectional_ranking_holdout",
    version="1",
    train_start=TRAIN_START,
    train_end=TRAIN_END,
    validation_start=VALIDATION_START,
    validation_end=VALIDATION_END,
    test_start=TEST_START,
    test_end=TEST_END,
)

database_url = f"sqlite+pysqlite:///{(repo_root / 'data' / 'swingtrader.sqlite').as_posix()}"
engine = resolve_database_engine(database_url=database_url)
bundle = build_temporal_dataset(engine=engine, spec=dataset_spec)
split_result = FixedTemporalSplitter(split_spec).assign(bundle)

bundle.manifest.to_manifest(), split_result.manifest.to_manifest()


## Select train and validation rows

The locked test is intentionally not read in this notebook. All three learned models use the same generated feature matrix and the same outer train/validation ranges.


In [ ]:
from swingtrader.modeling.datasets import to_tabular_dataset

relative_return_column = f"market_relative_forward_return_{HORIZON}d"
percentile_column = f"forward_return_{HORIZON}d_cross_sectional_percentile"
relevance_column = f"forward_return_{HORIZON}d_relevance_grade"

tabular = to_tabular_dataset(bundle)
train_positions = split_result.indices("train")
validation_positions = split_result.indices("validation")

X_train = tabular.X.iloc[train_positions]
X_validation = tabular.X.iloc[validation_positions]
relative_return_train = bundle.targets[relative_return_column].iloc[train_positions]
relative_return_validation = bundle.targets[relative_return_column].iloc[validation_positions]
percentile_train = bundle.targets[percentile_column].iloc[train_positions]
percentile_validation = bundle.targets[percentile_column].iloc[validation_positions]
relevance_train = bundle.targets[relevance_column].iloc[train_positions]
relevance_validation = bundle.targets[relevance_column].iloc[validation_positions]
classification_train = percentile_train.ge(TOP_QUANTILE_THRESHOLD).astype("int8")

{
    "train_rows": len(X_train),
    "validation_rows": len(X_validation),
    "feature_count": len(X_train.columns),
    "train_dates": X_train.index.get_level_values("trading_date").nunique(),
    "validation_dates": X_validation.index.get_level_values("trading_date").nunique(),
    "top_quantile_prevalence": classification_train.mean(),
}


## Fit the three XGBoost variants

These are deliberately compact starting parameters for comparison, not a tuned search space. XGBoost handles the retained feature missing values directly.


In [ ]:
common_parameters = {
    "n_estimators": 400,
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 20,
    "reg_lambda": 1.0,
    "tree_method": "hist",
    "random_state": RANDOM_SEED,
    "n_jobs": -1,
}

regressor = XGBRegressor(
    objective="reg:squarederror",
    eval_metric="rmse",
    **common_parameters,
)
classifier = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    **common_parameters,
)
ranker = XGBRanker(
    objective="rank:ndcg",
    eval_metric=f"ndcg@{TOP_K}",
    **common_parameters,
)


In [ ]:
regressor.fit(X_train, relative_return_train)
classifier.fit(X_train, classification_train)


`XGBRanker` requires rows belonging to the same query to be contiguous. Here one query is one provider and trading date.


In [ ]:
from swingtrader.modeling.training import prepare_xgboost_ranking_data

X_rank_train, relevance_rank_train, train_query_ids = prepare_xgboost_ranking_data(
    X_train,
    relevance_train,
)
ranker.fit(X_rank_train, relevance_rank_train, qid=train_query_ids)


## Score validation and compare ranking diagnostics

All outputs are treated as ranking scores. The random baseline uses the existing deterministic index-based score generator. Rank IC is Spearman correlation with continuous market-relative return; NDCG uses the ordinal relevance grades.


In [ ]:
from swingtrader.modeling.training import (
    deterministic_random_scores,
    evaluate_cross_sectional_scores,
)

validation_scores = {
    "random": deterministic_random_scores(X_validation.index, seed=RANDOM_SEED),
    "regression": pd.Series(
        regressor.predict(X_validation),
        index=X_validation.index,
        name="score",
    ),
    "classification": pd.Series(
        classifier.predict_proba(X_validation)[:, 1],
        index=X_validation.index,
        name="score",
    ),
}

X_rank_validation, _, _ = prepare_xgboost_ranking_data(
    X_validation,
    relevance_validation,
)
ranker_scores = pd.Series(
    ranker.predict(X_rank_validation),
    index=X_rank_validation.index,
    name="score",
).reindex(X_validation.index)
validation_scores["ranking"] = ranker_scores

summaries = {}
daily_results = {}
for model_name, scores in validation_scores.items():
    summaries[model_name], daily_results[model_name] = evaluate_cross_sectional_scores(
        scores,
        relevance_validation,
        relative_return_validation,
        top_k=TOP_K,
    )

comparison = pd.DataFrame(summaries).T
comparison


## Inspect stability by date

A model should not be judged from one aggregate. These plots expose the distribution of daily rank IC and top-k excess return across validation dates.


In [ ]:
rank_ic_by_model = pd.concat(
    {name: result["rank_ic"] for name, result in daily_results.items()},
    axis=1,
)
rank_ic_by_model.plot.box(figsize=(10, 5), title="Validation daily rank IC")
plt.axhline(0, linewidth=1)
plt.ylabel("Spearman correlation")
plt.show()


In [ ]:
top_k_excess_by_model = pd.concat(
    {name: result["top_k_excess_return"] for name, result in daily_results.items()},
    axis=1,
)
top_k_excess_by_model.plot.box(
    figsize=(10, 5),
    title=f"Validation top-{TOP_K} market-relative return",
)
plt.axhline(0, linewidth=1)
plt.ylabel("Return")
plt.show()


## Inspect one validation date

Choose a date to compare the ranked candidates with the outcomes used only for evaluation.


In [ ]:
inspection_date = X_validation.index.get_level_values("trading_date").max()
inspection = pd.DataFrame(
    {name: scores for name, scores in validation_scores.items()}
).join(
    bundle.targets.loc[X_validation.index, [
        relative_return_column,
        percentile_column,
        relevance_column,
    ]]
)
inspection.xs(inspection_date, level="trading_date").sort_values(
    "ranking",
    ascending=False,
).head(TOP_K)


## Optional feature-importance inspection

Gain importance is a model diagnostic, not proof that a feature is stable or causal.


In [ ]:
feature_importance = pd.Series(
    ranker.feature_importances_,
    index=X_rank_train.columns,
).sort_values(ascending=False)
feature_importance.head(20).sort_values().plot.barh(
    figsize=(9, 7),
    title="XGBRanker feature importance",
)
plt.xlabel("Importance")
plt.show()


## Save a study result

When you materially change dates, features, targets, or model parameters, export the completed notebook to HTML under `notebooks/exports/`. The export is ignored by Git and preserves what was run. Then restore this notebook to its generic base state before committing further notebook changes.

The next decision is empirical: does the ranker improve validation NDCG, rank IC, and top-k return over regression, classification, and random scoring consistently enough to justify a more formal implementation?
